# Add APIs to Multi-Turn Dialogues
#### Temporary files to add APIs to the multi-turn dialogues. This should be moved to a script once m3_benchmark scripts are setup.

In [5]:
import json
import os
from tqdm import tqdm
from collections import defaultdict
import random

In [6]:
multiturn_foldername="/proj/m3benchmark/bird-train/multi_turn/train_v6_0730"
output_foldername="/proj/m3benchmark/bird-train/multi_turn/train_rest_v6_0730"
rest_api_foldername="/proj/m3benchmark/invocable-api-datasets/rest/v1_0723/train"

## Add REST APIs to multi-turn dialogues

In [7]:
RETRIVER_SOFT_CLUSTERS=[
    {
        "name": "Education & Students",
        "description": "Topics related to schools, universities, student information, and education statistics.",
        "members": [
            "california_schools",
            "student_club",
            "student_loan",
            "university",
            "cs_semester",
            "computer_student",
            "college_completion"
        ]
    },
    {
        "name": "Movies, TV & Entertainment, Social Media",
        "description": "Topics focused on movies, episodes, characters, and streaming platforms.",
        "members": [
            "movie",
            "movie_platform",
            "movie_3",
            "movies_4",
            "simpson_episodes",
            "disney",
            "law_episode",
            "movielens",
            "social_media",
            "talkingdata",
            "music_tracker",
            "music_platform_2",
            "card_games",
            "video_games",
            "superhero"
        ]
    },
    {
        "name": "Sports & Athletics",
        "description": "Topics covering various sports, athletes, games, and statistics from different leagues.",
        "members": [
            "formula_1",
            "professional_basketball",
            "european_football_1",
            "european_football_2",
            "olympics",
            "ice_hockey_draft",
            "hockey",
            "soccer_2016"
        ]
    },
    {
        "name": "Retail, Sales & Commerce",
        "description": "Topics related to retail operations, customer purchases, inventory, and product ordering.",
        "members": [
            "retails",
            "retail_world",
            "retail_complains",
            "superstore",
            "regional_sales",
            "car_retails",
            "book_publishing_company",
            "sales",
            "sales_in_weather",
            "works_cycles"
        ]
    },
    {
        "name": "Food, Restaurants & Beverage",
        "description": "Topics concerning restaurants, food inspection, menu items, and beverages like beer.",
        "members": [
            "restaurant",
            "food_inspection",
            "food_inspection_2",
            "cookbook",
            "beer_factory",
            "craftbeer",
            "menu"
        ]
    },
    {
        "name": "Technology & Software",
        "description": "Topics focusing on codebases, software platforms, applications, and user interactions in tech.",
        "members": [
            "app_store",
            "codebase_community",
            "codebase_comments",
            "software_company",
            "image_and_language"
        ]
    },
    {
        "name": "Health & Medicine",
        "description": "Topics that pertain to health data, medical conditions, surveys, and synthetic patients.",
        "members": [
            "toxicology",
            "synthea",
            "mental_health_survey",
            "thrombosis_prediction",
            "genes"
        ]
    },
    {
        "name": "Geography & Demographics",
        "description": "Topics centered around population, regional information, countries, and life expectancy.",
        "members": [
            "mondial_geo",
            "world",
            "address",
            "world_development_indicators"
        ]
    },
    {
        "name": "Transportation & Mobility",
        "description": "Topics dealing with modes of transport such as trains, cars, bikes, and flights.",
        "members": [
            "airline",
            "trains",
            "bike_share_1",
            "cars",
            "shipping"
        ]
    },
    {
        "name": "Finance & Economics",
        "description": "Topics that revolve around financial data, transactions, donations, and currencies.",
        "members": [
            "financial",
            "donor",
            "coinmarketcap",
            "debit_card_specializing"
        ]
    },
    {
        "name": "Government, Law & Public Services, Human Capital",
        "description": "Topics concerning legislation, public databases, law enforcement, and civic institutions.",
        "members": [
            "legislator",
            "chicago_crime",
            "shooting",
            "public_review_platform",
            "human_resources"
        ]
    },
    {
        "name": "Literature, Language & Publishing",
        "description": "Topics focusing on books, authors, literary works, and linguistic corpora.",
        "members": [
            "books",
            "authors",
            "shakespeare",
            "language_corpus",
            "citeseer"
        ]
    }
]

In [8]:
def process_soft_clusters():
    domain_retrievers=defaultdict(list)
    for cluster in RETRIVER_SOFT_CLUSTERS:
        for domain in cluster["members"]:
            domain_retrievers[domain]=["clapnq-"+str(random.choices(i["members"],k=1)[0]).replace("_","") for i in random.choices(RETRIVER_SOFT_CLUSTERS, k=10)]
    return domain_retrievers

def convert_apis_dict(apis):
    ques_id_dict={}
    id_elem_dict={}
    for api in apis:
        ques_id_dict[api["input"]]=api["sample_id"]
        id_elem_dict[api["sample_id"]]=api
    return ques_id_dict, id_elem_dict

In [9]:
filenames=os.listdir(multiturn_foldername)
retrievers_per_domain = process_soft_clusters()
for filename in tqdm(filenames):
    domain=filename.split("_multiturn_bird.json")[0]
    api_filename=f"{domain}_nestful_format_bird.json"
    with open(f"{multiturn_foldername}/{filename}", 'r') as j:
        data = json.load(j)
    with open(f"{rest_api_foldername}/{api_filename}", 'r') as j:
        apis = json.load(j)
    ques_id_dict, id_elem_dict=convert_apis_dict(apis)
    for idx_item, item in enumerate(data):
        tool_list=[]        
        for idx_turn, turn in enumerate(item["turns"]):
            if isinstance(turn["gold_sequence"][0],list):
                if len(turn["gold_sequence"]) == 1:
                    turn["gold_sequence"]=turn["gold_sequence"][0]
                else:
                    import pdb
                    pdb.set_trace()
            for idx_hop, hop in enumerate(turn["gold_sequence"]):
                if hop["question_type"] == "API":
                    if hop["question"] not in ques_id_dict:
                        data[idx_item].update({"unanswerable":f"No API for turn {idx_turn+1} hop {idx_hop+1}"})
                        continue
                    sample_id=ques_id_dict[hop["question"]]
                    api_elem=id_elem_dict[sample_id]
                    assert api_elem["input"]==hop["question"]
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["OUTPUT_AFTER_EXECUTING_API"]=api_elem["OUTPUT_AFTER_EXECUTING_API"]
                    # data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["tools"]=api_elem["tools"] # Common tool list added
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["API"]=api_elem["API"]                    
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["output"]=api_elem["output"]
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["ignore"]=api_elem["ignore"]
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["parameterized_query"]=api_elem["parameterized_query"]
                    if len(tool_list)!=0:
                        if not set([tl["name"] for tl in tool_list]) == set([el["name"] for el in api_elem["tools"]]):
                            # These should be same.
                            import pdb
                            pdb.set_trace()
                    elif len(tool_list)==0:
                        tool_list=api_elem["tools"]
        data[idx_item]["retrievers"]=retrievers_per_domain[item["dataset_name"]]
        if len(tool_list)!=0:
            data[idx_item]["tools"]=tool_list

    with open(f'{output_foldername}/{filename}', 'w') as fout:
        json.dump(data , fout)

100%|██████████| 47/47 [02:42<00:00,  3.46s/it]
